# Pilot: Variance × Filter Analysis

Analyzes data from the **Pilot: Variance × Filter** experiment.

**Research question:** How do Exponential vs. One Euro filters compare at matched variance levels?

Upload 3 files:
- Raw trial data (`pilot-variance-filter-raw-data-*.csv`)
- Variance measurement (`pilot-variance-filter-variance-measurement-*.csv`)
- Pre-computed results (`pilot-variance-filter-results-*.csv`) — for reference only; we recompute with corrected Wₑ

All metrics (Wₑ, IDₑ, TP) are computed using the **ISO 9241-9 directional projection** method.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Upload your CSV files
RAW_DATA_FILE = 'pilot-variance-filter-raw-data-2026-02-17T22_44_34.435Z.csv'  # <-- change filename if needed
VARIANCE_FILE = 'pilot-variance-filter-variance-measurement-2026-02-17T22_44_34.435Z.csv'  # <-- change filename if needed
RESULTS_FILE = 'pilot-variance-filter-results-2026-02-17T22_44_34.435Z.csv'  # <-- change filename if needed

# Try loading from Downloads folder first, then current directory
import os
def load_csv(filename):
    paths = [
        filename,
        os.path.join(os.path.expanduser('~/Downloads'), filename),
    ]
    for p in paths:
        if os.path.exists(p):
            print(f'Loaded: {p}')
            return pd.read_csv(p)
    raise FileNotFoundError(f'Could not find {filename}')

raw_data = load_csv(RAW_DATA_FILE)
variance_data = load_csv(VARIANCE_FILE)
old_results = load_csv(RESULTS_FILE)

print(f'\nRaw data: {len(raw_data)} trials')
print(f'Variance measurements: {len(variance_data)} configurations')
print(f'Old results: {len(old_results)} conditions')

## 1. Variance Measurement Verification

Check whether the variance-matched pairs actually produced similar variance for both filters.

In [ ]:
print('=== Variance Measurement Results ===')
print(f'{"Pair":>4} {"Filter":>12} {"Rank":>5} {"Expected":>10} {"Measured":>10} {"Diff%":>8}')
print('-' * 55)
for _, row in variance_data.iterrows():
    print(f'{int(row["PairNumber"]):>4} {row["FilterType"]:>12} {int(row["FilterRank"]):>5} '
          f'{row["ExpectedVariance_px"]:>10.2f} {row["MeasuredVariance_px"]:>10.2f} '
          f'{row["DifferencePercent"]:>7.1f}%')

print('\n=== Pair-wise Variance Match ===')
for pair_num in variance_data['PairNumber'].unique():
    pair = variance_data[variance_data['PairNumber'] == pair_num]
    exp_var = pair[pair['FilterType'] == 'exponential']['MeasuredVariance_px'].values[0]
    oe_var = pair[pair['FilterType'] == 'oneEuro']['MeasuredVariance_px'].values[0]
    diff = abs(exp_var - oe_var)
    print(f'Pair {int(pair_num)}: Exponential={exp_var:.2f}px, OneEuro={oe_var:.2f}px, Difference={diff:.2f}px')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
pairs = variance_data['PairNumber'].unique()
x = np.arange(len(pairs))
width = 0.3

exp_measured = [variance_data[(variance_data['PairNumber']==p) & (variance_data['FilterType']=='exponential')]['MeasuredVariance_px'].values[0] for p in pairs]
oe_measured = [variance_data[(variance_data['PairNumber']==p) & (variance_data['FilterType']=='oneEuro')]['MeasuredVariance_px'].values[0] for p in pairs]
exp_expected = [variance_data[(variance_data['PairNumber']==p) & (variance_data['FilterType']=='exponential')]['ExpectedVariance_px'].values[0] for p in pairs]

ax.bar(x - width/2, exp_measured, width, label='Exponential (measured)', color='#3b82f6')
ax.bar(x + width/2, oe_measured, width, label='One Euro (measured)', color='#f97316')
ax.scatter(x, exp_expected, color='red', marker='_', s=200, linewidths=3, zorder=5, label='Expected variance')

ax.set_xlabel('Pair Number')
ax.set_ylabel('Measured Variance (px)')
ax.set_title('Variance Measurement: Expected vs. Measured')
ax.set_xticks(x)
ax.set_xticklabels([f'Pair {int(p)}' for p in pairs])
ax.legend()
plt.tight_layout()
plt.show()

## 2. Recompute Metrics with Corrected Wₑ

Using ISO 9241-9 directional projection (combined-coordinate method):
- Project each endpoint onto its movement direction: `pᵢ = xᵢ·cos(θ) + yᵢ·sin(θ)`
- `Wₑ = 4.133 × SD(projections)`

In [ ]:
def compute_fitts_metrics(df):
    """Compute ISO 9241-9 metrics using directional projection for Wₑ."""
    results = []
    
    grouping = ['PairNumber', 'PairVariance', 'PairDescription', 
                'FilterType', 'FilterRank', 'FilterVariance', 'FilterLatency',
                'TargetSize', 'Amplitude']
    grouping = [c for c in grouping if c in df.columns]
    
    for keys, group in df.groupby(grouping):
        valid = group[group['MovementTime'].notna() & (group['MovementTime'] > 0)]
        if len(valid) < 2:
            continue
        
        mean_mt = valid['MovementTime'].mean()
        Ae = valid['ActualAmplitude'].mean()
        
        theta_rad = np.radians(valid['Direction'].astype(float))
        # Reconstruct correct target positions from screen center
        # (CSV TargetX/Y is buggy — computed from startPoint instead of center)
        first_trial = group[group['TrialInLayout'] == 0].iloc[0]
        center_x, center_y = first_trial['StartX'], first_trial['StartY']
        correct_target_x = center_x + valid['Amplitude'] * np.cos(theta_rad)
        correct_target_y = center_y + valid['Amplitude'] * np.sin(theta_rad)
        dx = valid['SelectionX'] - correct_target_x
        dy = valid['SelectionY'] - correct_target_y
        projections = dx * np.cos(theta_rad) + dy * np.sin(theta_rad)
        SDx = projections.std(ddof=1)
        We = 4.133 * SDx
        
        IDe = np.log2(Ae / We + 1) if We > 0 else 0
        TP = IDe / mean_mt if mean_mt > 0 else 0
        
        row = dict(zip(grouping, keys))
        row.update({
            'N': len(valid),
            'MeanMT': mean_mt,
            'Ae': Ae,
            'We': We,
            'IDe': IDe,
            'TP': TP
        })
        results.append(row)
    
    return pd.DataFrame(results)

results = compute_fitts_metrics(raw_data)
print(f'Computed metrics for {len(results)} conditions')
display(results.round(4))

## 3. Throughput Comparison: Exponential vs. One Euro

In [ ]:
print('=== Throughput by Pair and Filter ===')
print(f'{"Pair":>4} {"Variance":>10} {"Filter":>12} {"Avg TP":>8} {"Avg MT":>8} {"Avg We":>8} {"Avg IDe":>8}')
print('-' * 70)

summary = results.groupby(['PairNumber', 'PairVariance', 'FilterType']).agg(
    AvgTP=('TP', 'mean'),
    AvgMT=('MeanMT', 'mean'),
    AvgWe=('We', 'mean'),
    AvgIDe=('IDe', 'mean')
).reset_index()

for _, row in summary.iterrows():
    print(f'{int(row["PairNumber"]):>4} {row["PairVariance"]:>10.2f} {row["FilterType"]:>12} '
          f'{row["AvgTP"]:>8.3f} {row["AvgMT"]:>8.3f} {row["AvgWe"]:>8.2f} {row["AvgIDe"]:>8.3f}')

print('\n=== Filter Comparison per Pair ===')
for pair_num in summary['PairNumber'].unique():
    pair = summary[summary['PairNumber'] == pair_num]
    exp_tp = pair[pair['FilterType'] == 'exponential']['AvgTP'].values[0]
    oe_tp = pair[pair['FilterType'] == 'oneEuro']['AvgTP'].values[0]
    exp_mt = pair[pair['FilterType'] == 'exponential']['AvgMT'].values[0]
    oe_mt = pair[pair['FilterType'] == 'oneEuro']['AvgMT'].values[0]
    better_tp = 'One Euro' if oe_tp > exp_tp else 'Exponential'
    tp_diff = abs(oe_tp - exp_tp) / min(oe_tp, exp_tp) * 100
    print(f'Pair {int(pair_num)}: {better_tp} has {tp_diff:.1f}% higher throughput '
          f'(Exp={exp_tp:.3f}, 1€={oe_tp:.3f} bps)')
    print(f'         Movement Time: Exp={exp_mt:.3f}s, 1€={oe_mt:.3f}s')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = sorted(summary['PairNumber'].unique())
x = np.arange(len(pairs))
width = 0.3

pair_labels = []
for p in pairs:
    pair_var = variance_data[variance_data['PairNumber'] == p]
    expected = pair_var['ExpectedVariance_px'].mean()
    pair_labels.append(f'Pair {int(p)}\nvariance ~{expected:.0f}px')

# Throughput
ax = axes[0]
exp_tp = [summary[(summary['PairNumber']==p) & (summary['FilterType']=='exponential')]['AvgTP'].values[0] for p in pairs]
oe_tp = [summary[(summary['PairNumber']==p) & (summary['FilterType']=='oneEuro')]['AvgTP'].values[0] for p in pairs]
ax.bar(x - width/2, exp_tp, width, label='Exponential', color='#3b82f6')
ax.bar(x + width/2, oe_tp, width, label='One Euro', color='#f97316')
ax.set_ylabel('Throughput (bits/s)')
ax.set_title('Throughput by Pair')
ax.set_xticks(x)
ax.set_xticklabels(pair_labels)
ax.legend()

# Movement Time
ax = axes[1]
exp_mt = [summary[(summary['PairNumber']==p) & (summary['FilterType']=='exponential')]['AvgMT'].values[0] for p in pairs]
oe_mt = [summary[(summary['PairNumber']==p) & (summary['FilterType']=='oneEuro')]['AvgMT'].values[0] for p in pairs]
ax.bar(x - width/2, exp_mt, width, label='Exponential', color='#3b82f6')
ax.bar(x + width/2, oe_mt, width, label='One Euro', color='#f97316')
ax.set_ylabel('Movement Time (s)')
ax.set_title('Movement Time by Pair')
ax.set_xticks(x)
ax.set_xticklabels(pair_labels)
ax.legend()

# Effective Width
ax = axes[2]
exp_we = [summary[(summary['PairNumber']==p) & (summary['FilterType']=='exponential')]['AvgWe'].values[0] for p in pairs]
oe_we = [summary[(summary['PairNumber']==p) & (summary['FilterType']=='oneEuro')]['AvgWe'].values[0] for p in pairs]
ax.bar(x - width/2, exp_we, width, label='Exponential', color='#3b82f6')
ax.bar(x + width/2, oe_we, width, label='One Euro', color='#f97316')
ax.set_ylabel('Effective Width (px)')
ax.set_title('Effective Width by Pair')
ax.set_xticks(x)
ax.set_xticklabels(pair_labels)
ax.legend()

plt.tight_layout()
plt.show()

## 4. Throughput by Target Size and Amplitude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By Target Size
ax = axes[0]
for ftype, color in [('exponential', '#3b82f6'), ('oneEuro', '#f97316')]:
    subset = results[results['FilterType'] == ftype]
    by_size = subset.groupby('TargetSize')['TP'].mean()
    ax.plot(by_size.index, by_size.values, 'o-', color=color, label=ftype, linewidth=2, markersize=8)
ax.set_xlabel('Target Size (px)')
ax.set_ylabel('Throughput (bits/s)')
ax.set_title('Throughput vs. Target Size')
ax.legend()

# By Amplitude
ax = axes[1]
for ftype, color in [('exponential', '#3b82f6'), ('oneEuro', '#f97316')]:
    subset = results[results['FilterType'] == ftype]
    by_amp = subset.groupby('Amplitude')['TP'].mean()
    ax.plot(by_amp.index, by_amp.values, 'o-', color=color, label=ftype, linewidth=2, markersize=8)
ax.set_xlabel('Amplitude (px)')
ax.set_ylabel('Throughput (bits/s)')
ax.set_title('Throughput vs. Amplitude')
ax.legend()

plt.tight_layout()
plt.show()

## 5. Fitts' Law Regression (IDₑ vs. MT)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for ftype, color, marker in [('exponential', '#3b82f6', 'o'), ('oneEuro', '#f97316', 's')]:
    subset = results[results['FilterType'] == ftype]
    ax.scatter(subset['IDe'], subset['MeanMT'], color=color, marker=marker, s=60, alpha=0.7, label=ftype)
    
    # Linear regression: MT = a + b * IDe
    if len(subset) > 1:
        coeffs = np.polyfit(subset['IDe'], subset['MeanMT'], 1)
        x_fit = np.linspace(subset['IDe'].min(), subset['IDe'].max(), 100)
        ax.plot(x_fit, np.polyval(coeffs, x_fit), '--', color=color, alpha=0.8,
                label=f'{ftype}: MT = {coeffs[1]:.2f} + {coeffs[0]:.2f}·IDe (R²={np.corrcoef(subset["IDe"], subset["MeanMT"])[0,1]**2:.3f})')

ax.set_xlabel('Effective Index of Difficulty IDₑ (bits)')
ax.set_ylabel('Movement Time (s)')
ax.set_title('Fitts\' Law: MT vs. IDₑ')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Movement Time Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, pair_num in enumerate(sorted(raw_data['PairNumber'].unique())):
    ax = axes[i]
    pair = raw_data[raw_data['PairNumber'] == pair_num]
    
    exp_mt = pair[pair['FilterType'] == 'exponential']['MovementTime'].dropna()
    oe_mt = pair[pair['FilterType'] == 'oneEuro']['MovementTime'].dropna()
    
    ax.hist(exp_mt, bins=15, alpha=0.6, color='#3b82f6', label=f'Exp (M={exp_mt.mean():.2f}s)')
    ax.hist(oe_mt, bins=15, alpha=0.6, color='#f97316', label=f'1€ (M={oe_mt.mean():.2f}s)')
    ax.set_xlabel('Movement Time (s)')
    ax.set_ylabel('Count')
    ax.set_title(f'Pair {int(pair_num)}')
    ax.legend(fontsize=9)

plt.suptitle('Movement Time Distributions by Pair', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. Overall Summary

In [ ]:
overall = results.groupby('FilterType').agg(
    AvgTP=('TP', 'mean'),
    StdTP=('TP', 'std'),
    AvgMT=('MeanMT', 'mean'),
    StdMT=('MeanMT', 'std'),
    AvgWe=('We', 'mean'),
    AvgIDe=('IDe', 'mean')
).reset_index()

print('=== Overall Summary ===')
print(f'{"Filter":>12} {"TP (bps)":>12} {"MT (s)":>12} {"We (px)":>10} {"IDe (bits)":>12}')
print('-' * 62)
for _, row in overall.iterrows():
    print(f'{row["FilterType"]:>12} {row["AvgTP"]:>5.3f}±{row["StdTP"]:.3f} '
          f'{row["AvgMT"]:>5.3f}±{row["StdMT"]:.3f} '
          f'{row["AvgWe"]:>10.2f} {row["AvgIDe"]:>12.3f}')

exp_tp = overall[overall['FilterType'] == 'exponential']['AvgTP'].values[0]
oe_tp = overall[overall['FilterType'] == 'oneEuro']['AvgTP'].values[0]
better = 'One Euro' if oe_tp > exp_tp else 'Exponential'
diff_pct = abs(oe_tp - exp_tp) / min(oe_tp, exp_tp) * 100
print(f'\n{better} filter has {diff_pct:.1f}% higher overall throughput.')